# In-The-Wild — Compact EDA
### Binary deepfake detection: real vs. fake speech

**Milestone 2 — Group Project**

This is a CPU-only, Colab/Kaggle-ready EDA of the **In-The-Wild** corpus
(Müller et al., 2022, [arXiv:2203.16263](https://arxiv.org/abs/2203.16263)),
loaded from the Hugging Face Hub:
[`mueller91/In-The-Wild`](https://huggingface.co/datasets/mueller91/In-The-Wild).
It is the "wild" counterpart to `ASVspoof_notebook.ipynb`: the same seed-42
discipline, integrity checks, sampled audio statistics and report-ready takeaways,
but a section structure driven by what this corpus actually contains.

**Scope and operating constraints**

- **One 8.16 GB ZIP download, no extraction.** The HF repo ships only
  `release_in_the_wild.zip`. The dataset card's `load_dataset()` recipe is stale
  (there is no loading script), so the notebook downloads the archive and reads its
  central directory plus a seeded sample of members in place.
- **Eval-only benchmark.** There is no official train/dev/test protocol. The notebook
  documents that status and *proposes* (does not execute) a split for the later
  PyTorch milestone.
- **Seeded sample.** Audio statistics use a label-stratified sample of **2,000 clips**
  (≈6.3% of 31,779). All randomness uses seed 42.
- **CPU-only.** No deep-learning framework is imported.
- **Budget.** One 8.16 GB download (≈5–15 min on Colab) plus 2,000 short decodes; target
  runtime is under one hour. No full extraction.
- **Top-to-bottom execution.** One guarded install cell, then metadata, tables, figures
  and a manifest export under `working/`.
- **Reference.** Müller, Czempin, Dieckmann, Froghyar & Böttinger (2022),
  *Does Audio Deepfake Detection Generalize?*, arXiv:2203.16263.

**Contents**

0. Setup, configuration, seed 42
1. Data source, packaging and integrity checks
2. Dataset overview — label mix, speakers, file-size accounting
3. Audio statistics from a seeded sample — duration, level, format, quality
4. Spectral examples — waveform and log-mel, real vs. fake
5. Augmentation decision table
6. Split strategy and leakage checks
7. Conclusions and requirement checklist

> **A note on `PLAN.md` vs. the pinned release.** `PLAN.md` anticipated
> `real/`/`fake/` folders of mp3 files with variable sample rates and no speaker
> metadata. The actual pinned release is a flat `release_in_the_wild/` directory of
> **16 kHz mono float32 WAV** files plus `meta.csv` (`file,speaker,label`) and
> `attribution.txt`. The notebook follows the observed data, verifies every claim,
> and explicitly records the differences in §1. It does not fabricate mp3 or
> variable-sample-rate findings.

## 0. Setup and configuration

The first cell installs only missing packages. Colab and Kaggle already ship most of
them. Audio bytes are decoded with `soundfile`; `librosa` is used only for the log-mel
examples. `datasets` is deliberately **not** used: the repository has no loading
script, and a ZIP cannot be streamed.

In [ ]:
import importlib.util
import os
import subprocess
import sys

os.environ["PYTHONHASHSEED"] = "42"

_REQUIRED = {
    "huggingface_hub": "huggingface_hub>=0.23",
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "soundfile": "soundfile>=0.12",
    "librosa": "librosa>=0.10",
    "matplotlib": "matplotlib>=3.7",
    "seaborn": "seaborn>=0.13",
    "tqdm": "tqdm>=4.66",
}
_missing = [
    spec for module, spec in _REQUIRED.items() if importlib.util.find_spec(module) is None
]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
else:
    print("All required packages are available.")

In [ ]:
import hashlib
import io
import math
import random
import re
import time
import warnings
import zipfile
from pathlib import Path, PurePosixPath

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import Markdown, display
from matplotlib.patches import Patch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# Render figures inline in Jupyter/Colab. No-op when the file is run as a plain script.
try:
    from IPython import get_ipython

    _ipython = get_ipython()
    if _ipython is not None:
        _ipython.run_line_magic("matplotlib", "inline")
except Exception:
    pass

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HF_REPO_ID = "mueller91/In-The-Wild"
HF_REVISION = "eee168f92c367f8c82ff2cf42b6f61e362fd6211"  # pinned at authoring time
ZIP_FILE = "release_in_the_wild.zip"
ARCHIVE_DIR = "release_in_the_wild"

SAMPLE_SIZE = 2_000  # clips decoded for the audio statistics
LABEL_ORDER = ["real", "fake"]
LABEL_COLORS = {"real": "#2a9d8f", "fake": "#e76f51"}
LABEL_SOURCE_MAP = {"bona-fide": "real", "spoof": "fake"}
LABEL_DISPLAY = {"real": "real (bona-fide)", "fake": "fake (spoof)"}

# Published figures, used only for cross-checks; all observed values are computed.
PUBLISHED_TOTAL = 31_779
PUBLISHED_REAL = 19_963
PUBLISHED_FAKE = 11_816
PUBLISHED_TOTAL_HOURS = 37.9
PUBLISHED_REAL_HOURS = 20.7
PUBLISHED_FAKE_HOURS = 17.2
PAPER_SPEAKERS = 58

N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 80
FMIN = 20
FMAX = 8_000

SPLITS = ("train", "val", "test")
SPLIT_TARGET = np.array([0.8, 0.1, 0.1])

WORK_DIR = Path.cwd() / "working"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(os.environ.get("INTHEWILD_HF_CACHE", Path.cwd() / "hf_cache"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_OVERRIDE = os.environ.get("INTHEWILD_ZIP_PATH")  # optional local archive for reuse/testing
START_TIME = time.perf_counter()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 110,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "font.family": "DejaVu Sans",
    }
)

print(f"Dataset: {HF_REPO_ID} @ {HF_REVISION[:8]}")
print(f"Seed: {SEED} | Audio sample: {SAMPLE_SIZE:,} clips")
print(f"Cache: {CACHE_DIR}")
print(f"Working directory: {WORK_DIR}")

In [ ]:
def show_table(frame: pd.DataFrame, caption: str, precision: int = 3) -> None:
    """Render a numbered table with a bold caption."""
    display(Markdown(f"**{caption}**"))
    display(frame.round(precision) if precision else frame)


def pct(numerator: float, denominator: float) -> float:
    return 100.0 * numerator / denominator if denominator else float("nan")


def human_bytes(num_bytes: float) -> str:
    value = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(value) < 1000.0 or unit == "TB":
            return f"{value:,.2f} {unit}"
        value /= 1000.0
    return f"{value:,.2f} TB"

## 1. Data source, packaging and integrity checks

The Hugging Face repository has no loading script and no Parquet shards — only
`release_in_the_wild.zip`. `hf_hub_download` retrieves the pinned archive once;
`zipfile` then reads the **central directory** without extracting anything. This keeps
disk use at ~8.16 GB instead of ~8.7 GB extracted, and lets the notebook enumerate
every member cheaply.

### Dataset reference card

| Fact | Value asserted by the release / paper | How the notebook verifies it |
|---|---|---|
| Packaging | one `release_in_the_wild.zip` (8.16 GB) | HF file listing + `Path.stat()` |
| Contents | 31,779 clips; 19,963 real / 11,816 fake | central directory + `meta.csv` |
| Duration | 37.9 h total (20.7 h real / 17.2 h fake) | sampled decode + file-size proxy |
| Speakers | paper: 58; release metadata: 54 names | `meta.csv` groupby |
| Labels | `bona-fide` / `spoof` → real / fake | `meta.csv` + assertions |
| Attacks | one undisclosed commercial TTS; no attack IDs | schema probe |
| Protocol | eval-only; no official train/dev/test | documentation + split proposal |
| Format | observed: WAV, IEEE float32, 16 kHz, mono | decoded sample |
| License | CC BY-SA 4.0 (HF card) | cited, not inferred |

### What the release actually contains — and how it differs from `PLAN.md`

- **Layout:** the archive is flat: `release_in_the_wild/<index>.wav`, plus
  `meta.csv` and `attribution.txt`. There are no `real/` or `fake/` folders; labels
  come from `meta.csv`, not folder membership.
- **Audio container:** files are uncompressed **WAV / IEEE float32**, not mp3. In the
  seeded sample all are **16 kHz mono**; the wildness is in the source material and
  recording conditions, not in a variable container.
- **Speaker metadata:** `meta.csv` has a `speaker` column, so per-speaker analysis and
  a speaker-disjoint split *are* possible. The folder/name convention does not encode
  labels or speakers.
- **Speaker count:** the paper reports 58 speakers; the released metadata exposes
  **54 distinct names**. The notebook uses the 54 observed names and records the
  discrepancy rather than silently copying 58.
- **Confirmed:** one undisclosed commercial TTS, no attack-ID axis, and no official
  split. These are documented, not computed.

In [ ]:
if ZIP_OVERRIDE:
    zip_path = Path(ZIP_OVERRIDE).expanduser().resolve()
    if not zip_path.exists():
        raise FileNotFoundError(f"INTHEWILD_ZIP_PATH does not exist: {zip_path}")
    repo_files = [ZIP_FILE]
    download_seconds = 0.0
    print(f"Using local archive override: {zip_path}")
else:
    repo_files = HfApi().list_repo_files(
        repo_id=HF_REPO_ID, repo_type="dataset", revision=HF_REVISION
    )
    if ZIP_FILE not in repo_files:
        raise FileNotFoundError(f"{ZIP_FILE!r} not found in {HF_REPO_ID}@{HF_REVISION}")
    print(f"Repo files at pinned revision: {', '.join(repo_files)}")
    download_start = time.perf_counter()
    zip_path = Path(
        hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=ZIP_FILE,
            repo_type="dataset",
            revision=HF_REVISION,
            cache_dir=str(CACHE_DIR),
        )
    )
    download_seconds = time.perf_counter() - download_start

archive_bytes = zip_path.stat().st_size
print(f"Archive: {zip_path}")
print(f"Downloaded/cached size: {human_bytes(archive_bytes)} ({archive_bytes:,} bytes)")
print(f"Archive retrieval time: {download_seconds / 60:.1f} min")

In [ ]:
with zipfile.ZipFile(zip_path) as zf:
    infos = zf.infolist()

    audio_infos = [
        info
        for info in infos
        if not info.is_dir() and PurePosixPath(info.filename).suffix.lower() == ".wav"
    ]
    meta_infos = [
        info for info in infos if PurePosixPath(info.filename).name == "meta.csv"
    ]
    attribution_infos = [
        info for info in infos if PurePosixPath(info.filename).name == "attribution.txt"
    ]
    dir_infos = [info for info in infos if info.is_dir()]
    known_names = {
        info.filename for info in audio_infos + meta_infos + attribution_infos + dir_infos
    }
    unexpected_infos = [
        info for info in infos if not info.is_dir() and info.filename not in known_names
    ]

    if len(meta_infos) != 1:
        raise ValueError(f"Expected exactly one meta.csv, found {len(meta_infos)}")
    if len(attribution_infos) != 1:
        raise ValueError(f"Expected exactly one attribution.txt, found {len(attribution_infos)}")
    if unexpected_infos:
        raise ValueError(f"Unexpected archive members: {[i.filename for i in unexpected_infos]}")
    if not audio_infos:
        raise ValueError("No .wav members found in the archive")

    audio_parents = {PurePosixPath(info.filename).parent.as_posix() for info in audio_infos}
    if audio_parents != {ARCHIVE_DIR}:
        raise ValueError(f"Unexpected audio parent folder(s): {sorted(audio_parents)}")
    audio_extensions = {PurePosixPath(info.filename).suffix.lower() for info in audio_infos}
    if audio_extensions != {".wav"}:
        raise ValueError(f"Unexpected audio extensions: {sorted(audio_extensions)}")

    meta_bytes = zf.read(meta_infos[0].filename)
    attribution_text = zf.read(attribution_infos[0].filename).decode("utf-8").strip()

meta = pd.read_csv(io.BytesIO(meta_bytes))
meta.columns = [str(column).strip() for column in meta.columns]
required_meta_columns = {"file", "speaker", "label"}
if not required_meta_columns.issubset(meta.columns):
    raise ValueError(
        f"meta.csv must contain {sorted(required_meta_columns)}, found {list(meta.columns)}"
    )

print(f"ZIP entries: {len(infos):,}")
print(f"  audio members: {len(audio_infos):,}")
print(f"  metadata:      {meta_infos[0].filename} ({human_bytes(meta_infos[0].file_size)})")
print(f"  attribution:   {attribution_infos[0].filename} ({attribution_infos[0].file_size} bytes)")
print(f"  directories:   {len(dir_infos)}")
print(f"Attribution: {attribution_text}")

In [ ]:
records = []
for info in audio_infos:
    member_path = info.filename
    file_name = PurePosixPath(member_path).name
    stem = PurePosixPath(file_name).stem
    if not re.fullmatch(r"\d+", stem):
        raise ValueError(f"Non-numeric clip stem: {member_path!r}")
    records.append(
        {
            "file_name": file_name,
            "file_index": int(stem),
            "member_path": member_path,
            "extension": PurePosixPath(file_name).suffix.lower(),
            "file_size": int(info.file_size),
            "compressed_size": int(info.compress_size),
            "crc32": f"{info.CRC:08x}",
        }
    )

manifest = pd.DataFrame.from_records(records)
if manifest.empty:
    raise ValueError("Audio manifest is empty")
if not manifest["file_name"].is_unique:
    raise ValueError("Duplicate audio basenames in the archive")
if set(manifest["file_index"]) != set(range(len(manifest))):
    raise ValueError("Clip indices are not dense 0..N-1; archive layout has changed")
manifest = manifest.sort_values("file_index").reset_index(drop=True)

if not meta["file"].is_unique:
    raise ValueError("Duplicate file rows in meta.csv")
extra_meta_columns = [column for column in meta.columns if column not in required_meta_columns]
manifest = manifest.merge(
    meta[["file", "speaker", "label"]],
    left_on="file_name",
    right_on="file",
    how="left",
    validate="one_to_one",
)
manifest = manifest.drop(columns=["file"])

meta_only = set(meta["file"]) - set(manifest["file_name"])
audio_only = set(manifest["file_name"]) - set(meta["file"])
if meta_only or audio_only:
    raise ValueError(f"Metadata/audio misalignment: meta-only={len(meta_only)}, audio-only={len(audio_only)}")

unexpected_labels = sorted(set(manifest["label"].dropna()) - set(LABEL_SOURCE_MAP))
if unexpected_labels:
    raise ValueError(f"Unexpected labels: {unexpected_labels}")

manifest["label_source"] = manifest["label"]
manifest["label"] = manifest["label"].map(LABEL_SOURCE_MAP)
manifest["target"] = (manifest["label"] == "fake").astype(int)
manifest["stem_pattern"] = manifest["file_name"].str.replace(r"\d+", "#", regex=True)

n_total = len(manifest)
n_real = int((manifest["label"] == "real").sum())
n_fake = int((manifest["label"] == "fake").sum())
n_speakers = int(manifest["speaker"].nunique())
n_missing_labels = int(manifest["label"].isna().sum())
n_missing_speakers = int(manifest["speaker"].isna().sum())
n_zero_byte = int((manifest["file_size"] == 0).sum())
uncompressed_bytes = int(manifest["file_size"].sum())
compressed_audio_bytes = int(manifest["compressed_size"].sum())
n_meta_speakers_expected = PAPER_SPEAKERS

manifest = manifest[
    [
        "file_name",
        "file_index",
        "member_path",
        "extension",
        "speaker",
        "label_source",
        "label",
        "target",
        "file_size",
        "compressed_size",
        "crc32",
        "stem_pattern",
    ]
].reset_index(drop=True)

print(f"Manifest: {n_total:,} clips | real={n_real:,} | fake={n_fake:,}")
print(f"Speakers: {n_speakers} | missing labels={n_missing_labels} | missing speakers={n_missing_speakers}")
print(f"Uncompressed audio: {human_bytes(uncompressed_bytes)} | compressed audio payload: {human_bytes(compressed_audio_bytes)}")
display(manifest[["file_name", "speaker", "label", "target", "file_size", "crc32"]].head(6))

In [ ]:
packaging = pd.DataFrame(
    [
        ("HF repo files (pinned)", ", ".join(repo_files)),
        ("Audio archives", "1 (`release_in_the_wild.zip`)"),
        ("Archive size (on disk)", f"{human_bytes(archive_bytes)} ({archive_bytes:,} bytes)"),
        ("ZIP entries", f"{len(infos):,}"),
        ("Audio members", f"{len(audio_infos):,} (`.wav`)"),
        ("Metadata member", f"`meta.csv` — {human_bytes(meta_infos[0].file_size)}"),
        ("Attribution member", f"`attribution.txt` — {attribution_infos[0].file_size} bytes"),
        ("Directory entries", f"{len(dir_infos)}"),
        ("Uncompressed audio payload", human_bytes(uncompressed_bytes)),
        ("Compressed audio payload", human_bytes(compressed_audio_bytes)),
        ("Compression ratio (uncompressed / archive)", f"{uncompressed_bytes / archive_bytes:.2f}×"),
        ("Archive retrieval", f"{download_seconds / 60:.1f} min" if not ZIP_OVERRIDE else "local override"),
    ],
    columns=["metric", "value"],
)
show_table(packaging, "Table 1.1 — HF packaging inventory", precision=0)

In [ ]:
schema_rows = [
    ("file", "yes", "string", f"{int(meta['file'].isna().sum()):,}", f"{meta['file'].nunique():,}", "`31778.wav`", "join key; exact basename"),
    ("speaker", "yes", "string", f"{n_missing_speakers:,}", f"{n_speakers:,}", "`Barack Obama`", "per-speaker analysis possible"),
    ("label", "yes", "string", f"{n_missing_labels:,}", f"{meta['label'].nunique():,}", "`bona-fide` / `spoof`", "mapped to `real` / `fake`"),
    ("source / video ID", "no", "—", "—", "—", "—", "source-level overlap cannot be audited"),
    ("attack ID", "no", "—", "—", "—", "—", "single undisclosed commercial TTS"),
    ("duration", "no", "—", "—", "—", "—", "derived from decoded frames / WAV header"),
    ("official split", "no", "—", "—", "—", "—", "eval-only benchmark"),
    ("license", "card", "—", "—", "—", "CC BY-SA 4.0", "HF dataset card; authors' site states Apache 2.0"),
]
schema = pd.DataFrame(
    schema_rows, columns=["field", "present", "type", "missing", "unique", "example", "note"]
)
show_table(schema, "Table 1.2 — Metadata schema and completeness", precision=0)

In [ ]:
integrity_rows = [
    ("Expected flat layout (`release_in_the_wild/*.wav` + metadata)", "observed", "PASS"),
    ("Audio members discovered", f"{n_total:,}", "PASS" if n_total == PUBLISHED_TOTAL else "REVIEW"),
    ("Real clips vs. published", f"{n_real:,} / {PUBLISHED_REAL:,}", "PASS" if n_real == PUBLISHED_REAL else "REVIEW"),
    ("Fake clips vs. published", f"{n_fake:,} / {PUBLISHED_FAKE:,}", "PASS" if n_fake == PUBLISHED_FAKE else "REVIEW"),
    ("Unique audio basenames", f"{manifest['file_name'].nunique():,} / {n_total:,}", "PASS" if manifest["file_name"].is_unique else "FAIL"),
    ("Dense numeric filenames 0..N-1", f"{manifest['file_index'].min()}..{manifest['file_index'].max()}", "PASS" if set(manifest["file_index"]) == set(range(n_total)) else "REVIEW"),
    ("Metadata rows", f"{len(meta):,}", "PASS" if len(meta) == n_total else "FAIL"),
    ("Metadata ↔ audio 1:1 alignment", f"{n_total - len(meta_only) - len(audio_only):,} / {n_total:,}", "PASS" if not meta_only and not audio_only else "FAIL"),
    ("Missing labels", f"{n_missing_labels:,}", "PASS" if n_missing_labels == 0 else "REVIEW"),
    ("Missing speaker names", f"{n_missing_speakers:,}", "PASS" if n_missing_speakers == 0 else "REVIEW"),
    ("Label values in allowed set", ", ".join(sorted(set(manifest["label"]))), "PASS" if set(manifest["label"]) == set(LABEL_ORDER) else "FAIL"),
    ("Media extension", ", ".join(sorted(set(manifest["extension"]))), "PASS" if set(manifest["extension"]) == {".wav"} else "FAIL"),
    ("Zero-byte audio members", f"{n_zero_byte:,}", "PASS" if n_zero_byte == 0 else "FAIL"),
    ("Unexpected archive members", f"{len(unexpected_infos):,}", "PASS" if not unexpected_infos else "REVIEW"),
    ("Distinct speaker names vs. paper", f"{n_speakers} / {n_meta_speakers_expected}", "REVIEW" if n_speakers != n_meta_speakers_expected else "PASS"),
    ("Attribution present", "yes", "PASS"),
]
integrity = pd.DataFrame(integrity_rows, columns=["check", "observed", "status"])
show_table(integrity, "Table 1.3 — Data integrity checks", precision=0)

**Takeaway.** The pinned release is a single 8.16 GB ZIP64 archive, not a
`load_dataset()`-ready repository. Its central directory exposes **31,779 WAV clips**
plus `meta.csv` and `attribution.txt`; no extraction is needed. The label counts match
the published **19,963 real / 11,816 fake** exactly, filenames are dense `0..31778`,
and metadata joins 1:1 to audio. The observed metadata exposes **54 speakers**, four
fewer than the paper's 58; that discrepancy is recorded rather than hidden. The
assumed `real/`/`fake/` mp3 layout in `PLAN.md` does not exist in this release.

## 2. Dataset overview

All overview tables and figures use the **full manifest** (31,779 clips), not the
2,000-clip sample. The corpus is real-majority: **62.8% real vs. 37.2% fake**, the
inverse direction of the ASVspoof 2019 LA eval set (89.7% spoof). A majority-class
classifier therefore already reaches ~62.8% accuracy, so accuracy alone is not a
meaningful headline metric. All fakes come from a single undisclosed commercial TTS;
the paper describes them as segments from 219 publicly available videos advertising
deepfakes, so there is no attack-ID axis to analyse.

In [ ]:
real_pct = pct(n_real, n_total)
fake_pct = pct(n_fake, n_total)
majority_acc = max(real_pct, fake_pct)

speaker_table = (
    manifest.groupby("speaker")
    .agg(
        clips=("file_name", "size"),
        real=("label", lambda values: int((values == "real").sum())),
        fake=("label", lambda values: int((values == "fake").sum())),
    )
    .sort_values("clips", ascending=False)
)
speaker_table["fake_share"] = speaker_table["fake"] / speaker_table["clips"]
speaker_table["clip_share"] = speaker_table["clips"] / n_total
speaker_table["cumulative_share"] = speaker_table["clip_share"].cumsum()
n_speakers_both = int(((speaker_table["real"] > 0) & (speaker_table["fake"] > 0)).sum())
top3_share = float(speaker_table["clips"].head(3).sum() / n_total)
top10_share = float(speaker_table["clips"].head(10).sum() / n_total)
clips_per_speaker_median = float(speaker_table["clips"].median())

overview = pd.DataFrame(
    [
        ("Clips", f"{n_total:,}"),
        ("Real (bona-fide)", f"{n_real:,} ({real_pct:.1f}%)"),
        ("Fake (spoof)", f"{n_fake:,} ({fake_pct:.1f}%)"),
        ("Majority-class accuracy", f"{majority_acc:.1f}%"),
        ("Speakers (released metadata)", f"{n_speakers}"),
        ("Speakers (paper)", f"{PAPER_SPEAKERS}"),
        ("Speakers with both labels", f"{n_speakers_both} / {n_speakers}"),
        ("Clips per speaker: min / median / max", f"{speaker_table['clips'].min():,} / {clips_per_speaker_median:,.1f} / {speaker_table['clips'].max():,}"),
        ("Top-3 speaker share", f"{top3_share * 100:.1f}%"),
        ("Top-10 speaker share", f"{top10_share * 100:.1f}%"),
        ("Published total duration", f"{PUBLISHED_TOTAL_HOURS:.1f} h"),
        ("Container (sampled in §3)", "WAV / IEEE float32"),
        ("Sample rate / channels (sampled in §3)", "16 kHz / mono"),
        ("Attack IDs", "none (single undisclosed commercial TTS)"),
        ("Official train/dev/test protocol", "none (eval-only benchmark)"),
        ("License (HF card)", "CC BY-SA 4.0"),
    ],
    columns=["metric", "value"],
)
show_table(overview, "Table 2.1 — Corpus at a glance", precision=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].pie(
    [n_real, n_fake],
    labels=[
        f"real\n{n_real:,} ({real_pct:.1f}%)",
        f"fake\n{n_fake:,} ({fake_pct:.1f}%)",
    ],
    colors=[LABEL_COLORS["real"], LABEL_COLORS["fake"]],
    startangle=90,
    counterclock=False,
    wedgeprops={"width": 0.42, "edgecolor": "white"},
    textprops={"fontsize": 10},
)
axes[0].set_title("(a) Label share")

baseline_values = [majority_acc, 50.0]
baseline_labels = ["majority-class\nbaseline", "balanced\nbaseline"]
bars = axes[1].bar(
    baseline_labels, baseline_values, color=["#e76f51", "#2a9d8f"], width=0.55
)
for bar, value in zip(bars, baseline_values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.1f}%",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )
axes[1].set_ylim(0, 105)
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("(b) Accuracy is misleading under imbalance")

fig.suptitle("Fig 2.1 — Class imbalance and the accuracy trap")
plt.tight_layout()
plt.show()

In [ ]:
speaker_display = speaker_table.copy()
speaker_display.index.name = "speaker"
speaker_display = speaker_display.rename(
    columns={
        "clips": "clips",
        "real": "real",
        "fake": "fake",
        "fake_share": "fake_share",
        "clip_share": "clip_share",
        "cumulative_share": "cumulative_share",
    }
)
show_table(
    speaker_display[["clips", "real", "fake", "fake_share", "clip_share", "cumulative_share"]],
    f"Table 2.2 — Per-speaker coverage (all {n_speakers} speakers)",
    precision=3,
)

In [ ]:
file_size_mb = manifest["file_size"] / 1e6
implied_mean_duration = PUBLISHED_TOTAL_HOURS * 3600 / n_total
implied_byte_rate = uncompressed_bytes / (PUBLISHED_TOTAL_HOURS * 3600)

file_accounting = pd.DataFrame(
    [
        ("Clips", f"{n_total:,}"),
        ("Total uncompressed audio", human_bytes(uncompressed_bytes)),
        ("Total compressed audio payload", human_bytes(compressed_audio_bytes)),
        ("Mean clip size", human_bytes(float(manifest["file_size"].mean()))),
        ("Median clip size", human_bytes(float(manifest["file_size"].median()))),
        ("Largest clip", human_bytes(float(manifest["file_size"].max()))),
        ("Published total duration (cross-check only)", f"{PUBLISHED_TOTAL_HOURS:.1f} h"),
        ("Implied mean clip duration from published hours", f"{implied_mean_duration:.2f} s"),
        ("Implied average byte rate from published hours", f"{implied_byte_rate:,.0f} B/s"),
        ("Expected byte rate for 16 kHz mono float32", "64,000 B/s (validated in §3)"),
    ],
    columns=["metric", "value"],
)
show_table(file_accounting, "Table 2.3 — File-size accounting and duration-proxy cross-check", precision=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

size_plot = manifest.assign(file_size_mb=file_size_mb)
sns.histplot(
    data=size_plot,
    x="file_size_mb",
    hue="label",
    hue_order=LABEL_ORDER,
    palette=LABEL_COLORS,
    bins=60,
    stat="density",
    common_norm=False,
    element="step",
    fill=True,
    alpha=0.25,
    log_scale=True,
    ax=axes[0],
)
axes[0].set_xlabel("Uncompressed clip size (MB, log scale)")
axes[0].set_ylabel("Density")
axes[0].set_title("(a) File-size distribution by label")
size_legend = [
    Patch(facecolor=LABEL_COLORS[label], edgecolor=LABEL_COLORS[label], alpha=0.35, label=label)
    for label in LABEL_ORDER
]
axes[0].legend(handles=size_legend, title="Label")

top_speakers = speaker_table.head(20).iloc[::-1]
axes[1].barh(top_speakers.index, top_speakers["clips"], color="#457b9d")
axes[1].set_xlabel("Clips")
axes[1].set_title("(b) Speaker concentration (top 20)")
axes[1].tick_params(axis="y", labelsize=7.5)
for index, (speaker, row) in enumerate(top_speakers.iterrows()):
    axes[1].text(
        row["clips"] + n_total * 0.002,
        index,
        f"{row['clips']:,} ({row['clip_share'] * 100:.1f}%)",
        va="center",
        fontsize=7,
    )
axes[1].set_xlim(0, top_speakers["clips"].max() * 1.25)

fig.suptitle("Fig 2.2 — File-size distribution and speaker concentration")
plt.tight_layout()
plt.show()

**Takeaway.** In-The-Wild is **real-majority** (19,963 real / 11,816 fake; 62.8% vs.
37.2%), so the majority-class baseline is 62.8% accuracy — the opposite imbalance
direction from ASVspoof 2019 LA eval. The 54 released speaker labels are highly
concentrated: the top 3 speakers account for 33.6% of clips and the top 10 for 70.6%,
and **every speaker contributes both real and fake clips**. This is a critical
leakage risk for any random split, and it is why §6 uses speaker-disjoint groups.

**Adequacy.** The corpus is adequate for an EDA and for a binary real/fake study with a
speaker-disjoint protocol: 37.9 h of wild English speech, 54 speakers, both labels per
speaker, and a realistic eval-only framing. It is **not** a substitute for multi-attack
lab training: all fakes come from one undisclosed commercial TTS, source/video IDs are
absent, and there is no official split. Report balanced accuracy, per-class recall,
AUC and EER rather than raw accuracy.

## 3. Audio statistics (seeded sample)

Decoding all 31,779 files would be unnecessary and memory-heavy. Instead, the notebook
draws a **label-stratified sample of 2,000 clips** with the seeded RNG, preserving the
real/fake mix to rounding, and decodes those members **directly from the ZIP**
(`zf.read(member)`), so no audio is extracted to disk. All statistics below are sample
estimates; the full-corpus duration is estimated separately by a size↔duration proxy
calibrated on the sample and is labelled as such.

In [ ]:
def stratified_sample(frame: pd.DataFrame, sample_size: int, seed: int) -> pd.DataFrame:
    """Seeded label-stratified sample with largest-remainder allocation."""
    rng = np.random.default_rng(seed)
    n_available = len(frame)
    n_target = min(sample_size, n_available)
    counts = frame["label"].value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)
    quotas = {label: n_target * counts[label] / n_available for label in LABEL_ORDER}
    allocation = {label: int(np.floor(quotas[label])) for label in LABEL_ORDER}
    remainder = n_target - sum(allocation.values())
    fractional = sorted(LABEL_ORDER, key=lambda label: quotas[label] - allocation[label], reverse=True)
    for label in fractional[:remainder]:
        allocation[label] += 1

    selected_indices = []
    for label in LABEL_ORDER:
        pool = frame.index[frame["label"] == label].to_numpy()
        n_pick = min(allocation[label], len(pool))
        if n_pick:
            selected_indices.extend(rng.choice(pool, size=n_pick, replace=False).tolist())
    sampled = frame.loc[selected_indices].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return sampled


sample_df = stratified_sample(manifest, SAMPLE_SIZE, SEED)
sample_allocation = sample_df["label"].value_counts().reindex(LABEL_ORDER)
print(f"Sample: {len(sample_df):,} / {n_total:,} clips ({pct(len(sample_df), n_total):.2f}%)")
print(
    "Allocation: "
    + " | ".join(
        f"{label.capitalize()} {int(sample_allocation[label]):,} "
        f"({pct(sample_allocation[label], len(sample_df)):.1f}%)"
        for label in LABEL_ORDER
    )
)
print(f"Full corpus: real {real_pct:.1f}% | fake {fake_pct:.1f}%")
display(
    sample_df[["file_name", "speaker", "label", "file_size", "crc32"]].head(5)
)

In [ ]:
def decode_audio_bytes(raw_bytes: bytes):
    """Decode one WAV member; fall back to librosa/audioread if soundfile fails."""
    info = sf.info(io.BytesIO(raw_bytes))
    audio, samplerate = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=False)
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio, int(samplerate), int(info.channels), str(info.format), str(info.subtype), "soundfile"


def analyze_audio_bytes(raw_bytes: bytes) -> dict:
    """Decode one clip and compute duration, level, crest, clipping, ZCR and silence."""
    try:
        audio, samplerate, channels, container, subtype, decoder = decode_audio_bytes(raw_bytes)
    except Exception:
        # Fallback: librosa may use a different backend, but is slower and less uniform.
        audio, samplerate = librosa.load(io.BytesIO(raw_bytes), sr=None, mono=False)
        audio = np.asarray(audio, dtype=np.float32)
        if audio.ndim > 1:
            audio = audio.mean(axis=0) if audio.shape[0] <= 2 else audio.mean(axis=1)
        samplerate = int(samplerate)
        channels = 1
        container, subtype, decoder = "unknown", "unknown", "librosa"

    audio = np.ascontiguousarray(audio, dtype=np.float32)
    if audio.size == 0 or samplerate <= 0:
        raise ValueError("empty audio payload or non-positive sample rate")
    if not np.isfinite(audio).all():
        raise ValueError("non-finite samples in decoded audio")

    record = {
        "duration": len(audio) / samplerate,
        "samplerate": samplerate,
        "channels": channels,
        "container": container,
        "subtype": subtype,
        "decoder": decoder,
        "peak_dbfs": np.nan,
        "rms_dbfs": np.nan,
        "crest_db": np.nan,
        "clipping_pct": np.nan,
        "silence_pct": np.nan,
        "zcr": np.nan,
    }

    peak = float(np.max(np.abs(audio)))
    rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
    record.update(
        {
            "peak_dbfs": 20 * math.log10(peak + 1e-12),
            "rms_dbfs": 20 * math.log10(rms + 1e-12),
            "crest_db": 20 * math.log10((peak + 1e-12) / (rms + 1e-12)),
            "clipping_pct": 100 * float(np.mean(np.abs(audio) >= 0.999)),
            "zcr": (
                float(np.mean(np.abs(np.diff(np.signbit(audio).astype(np.int8)))))
                if audio.size > 1
                else 0.0
            ),
        }
    )

    frame_length = max(1, int(0.02 * samplerate))
    hop_length = max(1, int(0.01 * samplerate))
    if audio.size >= frame_length:
        windows = np.lib.stride_tricks.sliding_window_view(audio, frame_length)[::hop_length]
        frame_rms = np.sqrt(np.mean(windows.astype(np.float64) ** 2, axis=1))
        record["silence_pct"] = 100 * float(np.mean(frame_rms < 1e-3))
    else:
        record["silence_pct"] = 0.0
    return record

In [ ]:
sample_records = []
decode_errors = []
with zipfile.ZipFile(zip_path) as zf:
    for row in tqdm(
        sample_df.itertuples(index=False),
        total=len(sample_df),
        desc="Decoding sampled clips",
    ):
        raw_bytes = zf.read(row.member_path)
        try:
            record = analyze_audio_bytes(raw_bytes)
        except Exception as exc:  # noqa: BLE001 - keep the loop running, report the count
            decode_errors.append({"file_name": row.file_name, "error": f"{type(exc).__name__}: {exc}"})
            record = {
                "duration": np.nan,
                "samplerate": np.nan,
                "channels": np.nan,
                "container": "error",
                "subtype": "error",
                "decoder": "error",
                "peak_dbfs": np.nan,
                "rms_dbfs": np.nan,
                "crest_db": np.nan,
                "clipping_pct": np.nan,
                "silence_pct": np.nan,
                "zcr": np.nan,
            }
        record.update(
            {
                "file_name": row.file_name,
                "md5": hashlib.md5(raw_bytes).hexdigest(),
            }
        )
        sample_records.append(record)

audio_sample = pd.DataFrame.from_records(sample_records)
audio_sample = audio_sample.merge(
    manifest[["file_name", "file_index", "speaker", "label", "target", "file_size"]],
    on="file_name",
    how="left",
    validate="one_to_one",
)
if audio_sample["label"].isna().any():
    raise ValueError("Sampled clips missing from the manifest")

print(f"Decoded: {len(audio_sample):,} | decode errors: {len(decode_errors)}")
print(f"Sample label mix: {audio_sample['label'].value_counts().reindex(LABEL_ORDER).to_dict()}")
if decode_errors:
    display(pd.DataFrame(decode_errors).head(10))

In [ ]:
duration_stats = (
    audio_sample.groupby("label")
    .agg(
        n=("duration", "size"),
        duration_mean_s=("duration", "mean"),
        duration_median_s=("duration", "median"),
        duration_p95_s=("duration", lambda values: values.quantile(0.95)),
        duration_max_s=("duration", "max"),
        peak_dbfs_median=("peak_dbfs", "median"),
        rms_dbfs_median=("rms_dbfs", "median"),
        crest_db_median=("crest_db", "median"),
        zcr_median=("zcr", "median"),
        silence_pct_median=("silence_pct", "median"),
        clipping_pct_mean=("clipping_pct", "mean"),
    )
    .reindex(LABEL_ORDER)
)
show_table(
    duration_stats.T,
    f"Table 3.1 — Duration and signal statistics (seeded sample, n={len(audio_sample):,})",
    precision=3,
)

sample_mean_hours = float(audio_sample["duration"].mean() * n_total / 3600)
sample_label_hours = (
    audio_sample.groupby("label")["duration"].mean()
    * pd.Series({"real": n_real, "fake": n_fake})
    / 3600
)
print(
    f"Sample-mean estimate of full-corpus duration: {sample_mean_hours:,.2f} h "
    f"(published: {PUBLISHED_TOTAL_HOURS:.1f} h)"
)
print(
    "  by label: "
    + " | ".join(
        f"{label} {sample_label_hours[label]:.2f} h "
        f"(published {PUBLISHED_REAL_HOURS if label == 'real' else PUBLISHED_FAKE_HOURS:.1f} h)"
        for label in LABEL_ORDER
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
sns.histplot(
    data=audio_sample,
    x="duration",
    hue="label",
    hue_order=LABEL_ORDER,
    palette=LABEL_COLORS,
    bins=60,
    stat="density",
    common_norm=False,
    element="step",
    fill=True,
    alpha=0.25,
    ax=ax,
)
for label in LABEL_ORDER:
    median = audio_sample.loc[audio_sample["label"] == label, "duration"].median()
    if pd.notna(median):
        ax.axvline(median, color=LABEL_COLORS[label], linestyle="--", linewidth=1)
ax.set_xlabel("Duration (s)")
ax.set_ylabel("Density")
ax.set_title(f"Fig 3.1 — Duration distribution by label (sample n={len(audio_sample):,})")
legend_handles = [
    Patch(facecolor=LABEL_COLORS[label], edgecolor=LABEL_COLORS[label], alpha=0.35, label=label)
    for label in LABEL_ORDER
]
ax.legend(handles=legend_handles, title="Label")
plt.tight_layout()
plt.show()

In [ ]:
level_metrics = [
    ("rms_dbfs", "RMS level (dBFS)"),
    ("zcr", "Zero-crossing rate"),
    ("silence_pct", "Silent frames (%)"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (metric, title) in zip(axes, level_metrics):
    sns.boxplot(
        data=audio_sample,
        x="label",
        y=metric,
        hue="label",
        order=LABEL_ORDER,
        hue_order=LABEL_ORDER,
        palette=LABEL_COLORS,
        showfliers=False,
        ax=ax,
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_title(title)
    ax.set_xlabel("Label")
    ax.set_ylabel("")
fig.suptitle("Fig 3.2 — Signal statistics by label (seeded sample)")
plt.tight_layout()
plt.show()

In [ ]:
audio_sample["codec"] = audio_sample["container"].astype(str) + " / " + audio_sample["subtype"].astype(str)
format_columns = [
    ("samplerate", "Sample rate (Hz)"),
    ("channels", "Channels"),
    ("codec", "Container / subtype"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (column, title) in zip(axes, format_columns):
    counts = audio_sample[column].astype(str).value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color="#457b9d")
    for bar, value in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + max(counts.values) * 0.02,
            f"{value:,}\n({pct(value, len(audio_sample)):.1f}%)",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    ax.set_title(title)
    ax.set_ylabel("Clips")
    ax.set_ylim(0, max(counts.values) * 1.22)
    ax.tick_params(axis="x", rotation=15)
fig.suptitle("Fig 3.3 — Format consistency in the seeded sample")
plt.tight_layout()
plt.show()

In [ ]:
sample_checks = pd.DataFrame(
    [
        {"check": "Clips decoded", "value": f"{len(audio_sample):,}", "status": "INFO"},
        {"check": "Decode errors", "value": f"{len(decode_errors):,}", "status": "PASS" if not decode_errors else "REVIEW"},
        {"check": "Sample rates observed", "value": ", ".join(f"{int(rate):,} Hz" for rate in sorted(audio_sample["samplerate"].dropna().unique())), "status": "PASS" if audio_sample["samplerate"].nunique() == 1 else "REVIEW"},
        {"check": "16 kHz share (%)", "value": f"{pct((audio_sample['samplerate'] == 16000).sum(), len(audio_sample)):.1f}", "status": "PASS" if audio_sample["samplerate"].eq(16000).all() else "REVIEW"},
        {"check": "Channels observed", "value": ", ".join(str(int(ch)) for ch in sorted(audio_sample["channels"].dropna().unique())), "status": "PASS" if audio_sample["channels"].nunique() == 1 else "REVIEW"},
        {"check": "Mono share (%)", "value": f"{pct(audio_sample['channels'].eq(1).sum(), len(audio_sample)):.1f}", "status": "PASS" if audio_sample["channels"].eq(1).all() else "REVIEW"},
        {"check": "Container / subtype observed", "value": ", ".join(sorted(audio_sample["codec"].unique())), "status": "PASS" if audio_sample["codec"].nunique() == 1 else "REVIEW"},
        {"check": "Clipped > 0.1% of samples", "value": f"{int((audio_sample['clipping_pct'] > 0.1).sum()):,}", "status": "PASS" if int((audio_sample['clipping_pct'] > 0.1).sum()) == 0 else "REVIEW"},
        {"check": "Silent > 50% of frames", "value": f"{int((audio_sample['silence_pct'] > 50).sum()):,}", "status": "PASS" if int((audio_sample['silence_pct'] > 50).sum()) == 0 else "REVIEW"},
        {"check": "Zero-length decodes", "value": f"{int(audio_sample['duration'].eq(0).sum()):,}", "status": "PASS" if int(audio_sample['duration'].eq(0).sum()) == 0 else "FAIL"},
        {"check": "Duration range (s)", "value": f"{audio_sample['duration'].min():.2f} – {audio_sample['duration'].max():.2f}", "status": "INFO"},
    ]
)
show_table(sample_checks, "Table 3.2 — Sample format and quality checks", precision=0)

In [ ]:
readable_sample = audio_sample.dropna(subset=["duration", "file_size"]).copy()
if len(readable_sample) < 10:
    raise ValueError("Not enough readable clips to calibrate the duration proxy")
fit_slope, fit_intercept = np.polyfit(readable_sample["duration"], readable_sample["file_size"], 1)
fit_predictions = fit_slope * readable_sample["duration"] + fit_intercept
ss_res = float(np.sum((readable_sample["file_size"] - fit_predictions) ** 2))
ss_tot = float(np.sum((readable_sample["file_size"] - readable_sample["file_size"].mean()) ** 2))
fit_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
manifest["duration_proxy_s"] = ((manifest["file_size"] - fit_intercept) / fit_slope).clip(lower=0)
readable_sample["duration_proxy_s"] = (
    (readable_sample["file_size"] - fit_intercept) / fit_slope
).clip(lower=0)
proxy_mae = float(np.mean(np.abs(readable_sample["duration"] - readable_sample["duration_proxy_s"])))

proxy_total_hours = float(manifest["duration_proxy_s"].sum() / 3600)
proxy_real_hours = float(manifest.loc[manifest["label"] == "real", "duration_proxy_s"].sum() / 3600)
proxy_fake_hours = float(manifest.loc[manifest["label"] == "fake", "duration_proxy_s"].sum() / 3600)
proxy_mean_duration = float(manifest["duration_proxy_s"].mean())
proxy_median_duration = float(manifest["duration_proxy_s"].median())
proxy_p95_duration = float(manifest["duration_proxy_s"].quantile(0.95))

duration_estimates = pd.DataFrame(
    [
        ("Sample mean × corpus counts", f"{sample_label_hours['real']:.2f}", f"{sample_label_hours['fake']:.2f}", f"{sample_mean_hours:.2f}", f"{sample_mean_hours - PUBLISHED_TOTAL_HOURS:+.2f}"),
        ("File-size proxy over full manifest", f"{proxy_real_hours:.2f}", f"{proxy_fake_hours:.2f}", f"{proxy_total_hours:.2f}", f"{proxy_total_hours - PUBLISHED_TOTAL_HOURS:+.2f}"),
        ("Published (HF card / paper)", f"{PUBLISHED_REAL_HOURS:.2f}", f"{PUBLISHED_FAKE_HOURS:.2f}", f"{PUBLISHED_TOTAL_HOURS:.2f}", "0.00"),
    ],
    columns=["estimate", "real_hours", "fake_hours", "total_hours", "delta_vs_published_h"],
)
show_table(duration_estimates, "Table 3.3 — Full-corpus duration estimates", precision=2)
print(
    f"Duration proxy fit: bytes = {fit_slope:,.0f} × seconds + {fit_intercept:,.0f} "
    f"(R²={fit_r2:.6f}, sample MAE={proxy_mae:.3f} s)"
)
print(
    f"Proxy duration: mean {proxy_mean_duration:.3f} s | median {proxy_median_duration:.3f} s "
    f"| p95 {proxy_p95_duration:.3f} s | max {manifest['duration_proxy_s'].max():.3f} s"
)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
sns.scatterplot(
    data=readable_sample,
    x="duration",
    y="file_size",
    hue="label",
    hue_order=LABEL_ORDER,
    palette=LABEL_COLORS,
    s=14,
    alpha=0.55,
    ax=ax,
)
line_x = np.linspace(0, readable_sample["duration"].max() * 1.02, 100)
ax.plot(line_x, fit_slope * line_x + fit_intercept, color="black", linewidth=1.5, label="linear fit")
ax.set_xlabel("Measured duration (s)")
ax.set_ylabel("Uncompressed WAV size (bytes)")
ax.set_title("Fig 3.4 — File-size ↔ duration proxy (sample fit)")
ax.legend(title="Label")
ax.text(
    0.98,
    0.05,
    f"R² = {fit_r2:.6f}\nMAE = {proxy_mae:.3f} s",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9,
    bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)
plt.tight_layout()
plt.show()

In [ ]:
sample_representativeness = pd.DataFrame(
    [
        ("Clips", f"{len(audio_sample):,}", f"{n_total:,}"),
        ("Real share (%)", f"{pct((audio_sample['label'] == 'real').sum(), len(audio_sample)):.1f}", f"{real_pct:.1f}"),
        ("Fake share (%)", f"{pct((audio_sample['label'] == 'fake').sum(), len(audio_sample)):.1f}", f"{fake_pct:.1f}"),
        ("Speakers", f"{audio_sample['speaker'].nunique():,}", f"{n_speakers:,}"),
        ("Clips per speaker: min / median / max", f"{audio_sample.groupby('speaker').size().min()} / {audio_sample.groupby('speaker').size().median():.1f} / {audio_sample.groupby('speaker').size().max()}", f"{speaker_table['clips'].min()} / {speaker_table['clips'].median():.1f} / {speaker_table['clips'].max()}"),
        ("Duration mean (s)", f"{audio_sample['duration'].mean():.2f}", f"{proxy_mean_duration:.2f}"),
        ("Duration median (s)", f"{audio_sample['duration'].median():.2f}", f"{proxy_median_duration:.2f}"),
        ("Duration p95 (s)", f"{audio_sample['duration'].quantile(0.95):.2f}", f"{proxy_p95_duration:.2f}"),
    ],
    columns=["metric", "seeded sample", "full corpus (proxy/metadata)"],
)
show_table(sample_representativeness, "Table 3.4 — Sample representativeness", precision=2)

**Takeaway.** The sampled audio is uniformly **16 kHz mono WAV / IEEE float32**
(Table 3.2, Fig 3.3), not mp3 with variable rates as `PLAN.md` anticipated. There are
no decode errors and the label-stratified sample preserves the corpus mix by
construction. Duration is strongly right-skewed: the full-manifest size proxy gives a
mean of ≈4.3 s, median ≈3.3 s, p95 ≈10.8 s and max ≈25.0 s, totalling **≈37.9 h** —
within a few minutes of the published figure. Loudness, crest factor, ZCR
and pause fraction vary between clips (Table 3.1, Fig 3.2), so training must use
fixed-length cropping/padding and a documented normalisation policy. The wildness of
this corpus is in its source material and recording conditions, not in container or
sample-rate inconsistency.

## 4. Spectral examples

One seeded real and one seeded fake clip from the sample, shown as waveform and
log-mel spectrogram. This is a qualitative sanity check, not a classifier: the two
classes share the same 16 kHz band, and with a single TTS the fake examples may share
artefacts that are subtle in a single spectrogram. The per-file `fmax` is clipped to
`min(FMAX, sr // 2)`, so the display remains valid if a future release contains a
lower sample rate.

In [ ]:
example_rows = {}
for offset, label in enumerate(LABEL_ORDER):
    candidates = readable_sample.loc[readable_sample["label"] == label]
    if candidates.empty:
        raise ValueError(f"No readable {label} example in the sample")
    example_rows[label] = candidates.sample(n=1, random_state=SEED + offset).iloc[0]

example_details = []
fig, axes = plt.subplots(2, 2, figsize=(15, 7))
with zipfile.ZipFile(zip_path) as zf:
    for row_index, label in enumerate(LABEL_ORDER):
        row = example_rows[label]
        member_path = manifest.loc[
            manifest["file_name"] == row["file_name"], "member_path"
        ].iloc[0]
        raw_bytes = zf.read(member_path)
        audio, samplerate, channels, container, subtype, decoder = decode_audio_bytes(raw_bytes)
        time_axis = np.arange(len(audio)) / samplerate
        fmax = min(FMAX, samplerate // 2)

        example_details.append(
            {
                "label": label,
                "file_name": row["file_name"],
                "speaker": row["speaker"],
                "duration_s": len(audio) / samplerate,
                "samplerate": samplerate,
                "channels": channels,
                "codec": f"{container} / {subtype}",
                "file_size_kb": row["file_size"] / 1000,
            }
        )

        waveform_ax, mel_ax = axes[row_index]
        waveform_ax.plot(time_axis, audio, color=LABEL_COLORS[label], linewidth=0.6)
        waveform_ax.set_xlim(0, time_axis[-1] if len(time_axis) else 0)
        amplitude_limit = max(1.0, float(np.max(np.abs(audio))) * 1.05) if len(audio) else 1.0
        waveform_ax.set_ylim(-amplitude_limit, amplitude_limit)
        waveform_ax.set_title(
            f"{LABEL_DISPLAY[label]} | {row['file_name']} | {row['speaker']} "
            f"| {len(audio) / samplerate:.2f} s"
        )
        waveform_ax.set_xlabel("Time (s)")
        waveform_ax.set_ylabel("Amplitude")

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=samplerate,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=fmax,
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(
            mel_db,
            sr=samplerate,
            hop_length=HOP_LENGTH,
            x_axis="time",
            y_axis="mel",
            fmax=fmax,
            cmap="magma",
            ax=mel_ax,
        )
        mel_ax.set_title(f"Log-mel — {LABEL_DISPLAY[label]} | {row['speaker']}")
        mel_ax.set_xlabel("Time (s)")
        mel_ax.set_ylabel("Mel frequency (Hz)")

fig.suptitle("Fig 4.1 — Waveform and log-mel examples (seeded sample)")
plt.tight_layout()
plt.show()

show_table(
    pd.DataFrame(example_details),
    "Table 4.1 — Seeded example clips used in Fig 4.1",
    precision=2,
)

**Takeaway.** Real and fake speech occupy the same 16 kHz band and have similar
time-frequency structure; no single visual cue separates them. The examples are useful
for checking preprocessing (mono, fixed-length crops, mel geometry), not for visual
classification. With one TTS, a model can learn generator-specific artefacts rather
than general deepfake cues, which is why the split and leakage checks in §6 matter
more than a single spectrogram.

## 5. Augmentation decision table

Augmentation is a training-pipeline concern, not an EDA deliverable, so no
transformations are applied here. The menu is shaped by this corpus: the released audio
is WAV, but the source material is wild online speech with unknown codec history, so a
**codec round-trip** is more relevant than in the clean ASVspoof LA setting. Apply every
waveform augmentation symmetrically to both labels and never change the label.

In [ ]:
augmentation_plan = pd.DataFrame(
    [
        {
            "augmentation": "AWGN",
            "domain": "waveform",
            "setting": "SNR 15 dB, random segment",
            "priority": "high",
            "risk": "medium",
            "why_it_matters": "matches noisy wild channels; cheap regularisation",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "SpecAugment",
            "domain": "log-mel",
            "setting": "2 time + 2 frequency masks",
            "priority": "high",
            "risk": "medium",
            "why_it_matters": "reduces overfitting to one TTS's spectral artefacts",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "mp3 round-trip",
            "domain": "waveform",
            "setting": "128 kbps / 64 kbps",
            "priority": "high",
            "risk": "medium",
            "why_it_matters": "source videos are lossy even though release is WAV",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "Opus round-trip",
            "domain": "waveform",
            "setting": "24 kbps, 16 kHz",
            "priority": "medium",
            "risk": "medium",
            "why_it_matters": "telephony/streaming channel simulation",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "Time shift",
            "domain": "waveform",
            "setting": "±0.15 s",
            "priority": "medium",
            "risk": "low",
            "why_it_matters": "cheap translation invariance; avoid circular wrap",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "Speed perturbation",
            "domain": "waveform",
            "setting": "0.90× / 1.10×",
            "priority": "medium",
            "risk": "medium",
            "why_it_matters": "speaker/tempo variation; changes duration and formants",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "Pitch perturbation",
            "domain": "waveform",
            "setting": "±2 semitones",
            "priority": "medium",
            "risk": "medium",
            "why_it_matters": "adds speaker diversity; keep shifts small",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "µ-law companding",
            "domain": "waveform",
            "setting": "μ = 255",
            "priority": "low",
            "risk": "medium",
            "why_it_matters": "cheap telephony proxy, not a real codec",
            "label_policy": "symmetric",
        },
        {
            "augmentation": "Synthetic reverb",
            "domain": "waveform",
            "setting": "RT60 ≈ 0.30 s",
            "priority": "low",
            "risk": "high",
            "why_it_matters": "already wild; heavy reverb may erase TTS cues",
            "label_policy": "symmetric",
        },
    ]
)
show_table(
    augmentation_plan,
    "Table 5.1 — Augmentation menu, setting, priority and risk",
    precision=0,
)

In [ ]:
score_map = {"low": 1, "medium": 2, "high": 3}
augmentation_scores = augmentation_plan.assign(
    priority_score=augmentation_plan["priority"].map(score_map),
    risk_score=augmentation_plan["risk"].map(score_map),
)
y_positions = np.arange(len(augmentation_scores))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), sharey=True)
axes[0].barh(y_positions, augmentation_scores["priority_score"], color="#2a9d8f")
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(augmentation_scores["augmentation"])
axes[0].invert_yaxis()
axes[0].set_xlim(0, 3.3)
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(["low", "medium", "high"])
axes[0].set_xlabel("Priority")
axes[0].set_title("(a) Priority")
for position, value in zip(y_positions, augmentation_scores["priority_score"]):
    axes[0].text(value + 0.05, position, f"{value}", va="center", fontsize=8)

axes[1].barh(y_positions, augmentation_scores["risk_score"], color="#e76f51")
axes[1].set_xlim(0, 3.3)
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(["low", "medium", "high"])
axes[1].set_xlabel("Risk")
axes[1].set_title("(b) Risk")
for position, value in zip(y_positions, augmentation_scores["risk_score"]):
    axes[1].text(value + 0.05, position, f"{value}", va="center", fontsize=8)

fig.suptitle("Fig 5.1 — Augmentation priority vs. risk")
plt.tight_layout()
plt.show()

**Takeaway.** Start with AWGN, SpecAugment and **codec round-trips**: they target the
wild-channel and lossy-source conditions that dominate this corpus, and they are cheap
on CPU. Add time/speed/pitch perturbation only after the baseline is stable; keep
reverb low because the data is already wild and heavy reverb can erase the very TTS
cues a detector needs. Apply waveform augmentations before log-mel extraction and
SpecAugment on the log-mel itself; the PyTorch milestone owns the implementation.

## 6. Split strategy and leakage checks

**Scope note.** In-The-Wild is an **evaluation-only** benchmark: there is no official
train/dev/test protocol and no official speaker-independent partition. Any split we
propose is a project choice, not a reproduction of the paper's evaluation. The notebook
therefore *defines* a split for the later PyTorch milestone and *does not execute* it:
no file-level split column is exported and no audio is copied.

### Speaker probe

`PLAN.md` warned that speaker metadata might be absent. It is present in `meta.csv`, so
the probe below checks completeness and cardinality rather than assuming it. Because
**all speakers contribute both labels**, a random clip-level split would leak speaker
identity across train/val/test. The proposal is therefore **speaker-disjoint**:
every clip from a speaker goes to exactly one split.

If a future release omitted speaker metadata, the documented fallback is a seeded
label-stratified 80/10/10 clip split with an explicit caveat that speaker overlap — and
therefore leakage — cannot be excluded.

In [ ]:
speaker_column = "speaker" if "speaker" in manifest.columns else None
speaker_missing_pct = (
    pct(manifest[speaker_column].isna().sum(), n_total) if speaker_column else 100.0
)
if speaker_column is not None and speaker_missing_pct > 10.0:
    speaker_column = None

speaker_series = manifest[speaker_column] if speaker_column else pd.Series(dtype=object)
speaker_clip_counts = speaker_series.dropna().value_counts()
speaker_both_count = int(
    (
        manifest.dropna(subset=[speaker_column])
        .groupby(speaker_column)["label"]
        .nunique()
        .eq(2)
        .sum()
    )
    if speaker_column
    else 0
)
speaker_probe = pd.DataFrame(
    [
        ("Speaker column present", "yes" if speaker_column else "no", "PASS" if speaker_column else "REVIEW"),
        ("Missing speaker values (%)", f"{speaker_missing_pct:.1f}", "PASS" if speaker_missing_pct == 0 else "REVIEW"),
        ("Distinct speaker names", f"{speaker_clip_counts.size:,}", "PASS" if speaker_clip_counts.size else "REVIEW"),
        ("Clips per speaker: min / median / max", f"{speaker_clip_counts.min():,} / {speaker_clip_counts.median():,.1f} / {speaker_clip_counts.max():,}" if len(speaker_clip_counts) else "—", "INFO"),
        ("Speakers with both labels", f"{speaker_both_count:,} / {speaker_clip_counts.size:,}", "REVIEW" if speaker_both_count else "INFO"),
        ("Speakers with <10 clips", f"{int((speaker_clip_counts < 10).sum()):,}", "INFO"),
        ("Top-3 speaker share", f"{top3_share * 100:.1f}%", "INFO"),
        ("Distinct names vs. paper (58)", f"{speaker_clip_counts.size:,} / {PAPER_SPEAKERS}", "REVIEW" if speaker_clip_counts.size != PAPER_SPEAKERS else "PASS"),
        ("Source / video ID column", "absent", "REVIEW"),
        ("Attack ID column", "absent", "INFO"),
    ],
    columns=["check", "observed", "status"],
)
show_table(speaker_probe, "Table 6.1 — Speaker metadata probe", precision=0)

In [ ]:
fingerprint_groups = manifest.groupby(["crc32", "file_size"]).agg(
    files=("file_name", "size"), labels=("label", "nunique")
)
duplicate_fingerprints = fingerprint_groups[fingerprint_groups["files"] > 1]
cross_label_fingerprints = duplicate_fingerprints[duplicate_fingerprints["labels"] > 1]
n_duplicate_files = int(duplicate_fingerprints["files"].sum())
n_cross_label_duplicate_files = int(cross_label_fingerprints["files"].sum())

md5_counts = audio_sample["md5"].value_counts()
md5_duplicate_groups = int((md5_counts > 1).sum())
md5_duplicate_files = int(md5_counts[md5_counts > 1].sum())

basename_duplicates = int(manifest["file_name"].duplicated().sum())

size_label_counts = manifest.groupby("file_size")["label"].nunique()
size_collision_groups = int((size_label_counts > 1).sum())
size_collision_files = int(
    manifest["file_size"].isin(size_label_counts[size_label_counts > 1].index).sum()
)

pattern_label_counts = manifest.groupby("stem_pattern")["label"].nunique()
pattern_overlap = int((pattern_label_counts > 1).sum())
pattern_overlap_pct = pct(pattern_overlap, len(pattern_label_counts))

# The leakage table is assembled after the split proposal, because its
# "speaker overlap across proposed splits" row needs the actual speaker sets.

In [ ]:
global_fake_share = n_fake / n_total
speaker_plot = speaker_table.sort_values("clips", ascending=False)
y_positions = np.arange(len(speaker_plot))

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 12),
    sharey=True,
    gridspec_kw={"width_ratios": [1.25, 1.0]},
)
axes[0].barh(y_positions, speaker_plot["real"], color=LABEL_COLORS["real"], label="real")
axes[0].barh(
    y_positions,
    speaker_plot["fake"],
    left=speaker_plot["real"],
    color=LABEL_COLORS["fake"],
    label="fake",
)
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(speaker_plot.index, fontsize=7)
axes[0].invert_yaxis()
axes[0].set_xlabel("Clips")
axes[0].set_title("(a) Real/fake counts per speaker")
axes[0].legend(title="Label", loc="lower right")

axes[1].scatter(
    speaker_plot["fake_share"],
    y_positions,
    color=LABEL_COLORS["fake"],
    s=22,
    zorder=3,
)
axes[1].axvline(global_fake_share, color="black", linestyle="--", linewidth=1, label="global fake share")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Fake share within speaker")
axes[1].set_title("(b) Fake share per speaker")
axes[1].legend(loc="lower right")

fig.suptitle("Fig 6.1 — Every speaker contributes both real and fake clips")
plt.tight_layout()
plt.show()

In [ ]:
def propose_speaker_disjoint_split(
    frame: pd.DataFrame, seed: int = SEED
) -> tuple[dict, pd.DataFrame, pd.DataFrame, float]:
    """Greedy speaker assignment + pairwise-swap local search for a 80/10/10 target."""
    speaker_stats = (
        frame.dropna(subset=["speaker"])
        .groupby("speaker")
        .agg(
            clips=("file_name", "size"),
            real=("label", lambda values: int((values == "real").sum())),
            fake=("label", lambda values: int((values == "fake").sum())),
            hours=("duration_proxy_s", "sum"),
        )
    )
    n_clips = int(speaker_stats["clips"].sum())
    n_fake_total = int(speaker_stats["fake"].sum())

    speakers = sorted(speaker_stats.index.tolist())
    rng = random.Random(seed)
    rng.shuffle(speakers)
    speakers.sort(key=lambda speaker: -int(speaker_stats.loc[speaker, "clips"]))

    vectors = {
        speaker: np.array(
            [
                speaker_stats.loc[speaker, "clips"],
                speaker_stats.loc[speaker, "real"],
                speaker_stats.loc[speaker, "fake"],
            ],
            dtype=np.int64,
        )
        for speaker in speakers
    }

    def split_cost(counts: np.ndarray) -> float:
        clip_share = counts[:, 0] / max(n_clips, 1)
        fake_alloc = counts[:, 2] / max(n_fake_total, 1)
        return float(
            ((clip_share - SPLIT_TARGET) ** 2).sum()
            + ((fake_alloc - SPLIT_TARGET) ** 2).sum()
        )

    counts = np.zeros((3, 3), dtype=np.int64)
    assignment = {}
    for speaker in speakers:
        best_split = None
        best_cost = None
        for split_index in range(3):
            trial = counts.copy()
            trial[split_index] += vectors[speaker]
            cost = split_cost(trial)
            if best_cost is None or cost < best_cost - 1e-15:
                best_split = split_index
                best_cost = cost
        assignment[speaker] = best_split
        counts[best_split] += vectors[speaker]

    # Guarantee that every split contains at least one speaker when possible.
    for split_index in range(3):
        if counts[split_index, 0] == 0:
            source = int(np.argmax(counts[:, 0]))
            candidates = [s for s in speakers if assignment[s] == source]
            if len(candidates) <= 1:
                continue
            candidate = min(candidates, key=lambda s: int(speaker_stats.loc[s, "clips"]))
            counts[source] -= vectors[candidate]
            assignment[candidate] = split_index
            counts[split_index] += vectors[candidate]

    def total_cost(current_assignment: dict) -> float:
        current_counts = np.zeros((3, 3), dtype=np.int64)
        for speaker, split_index in current_assignment.items():
            current_counts[split_index] += vectors[speaker]
        return split_cost(current_counts)

    current_cost = total_cost(assignment)
    for _ in range(100):
        improved = False
        pairs = [
            (first, second)
            for index, first in enumerate(speakers)
            for second in speakers[index + 1 :]
            if assignment[first] != assignment[second]
        ]
        rng.shuffle(pairs)
        for first, second in pairs:
            first_split, second_split = assignment[first], assignment[second]
            assignment[first], assignment[second] = second_split, first_split
            new_cost = total_cost(assignment)
            if new_cost < current_cost - 1e-14:
                current_cost = new_cost
                improved = True
            else:
                assignment[first], assignment[second] = first_split, second_split
        if not improved:
            break

    proposed = speaker_stats.copy()
    proposed["split"] = proposed.index.map(assignment).map(dict(enumerate(SPLITS)))
    split_table = (
        proposed.groupby("split")
        .agg(
            speakers=("clips", "size"),
            clips=("clips", "sum"),
            real=("real", "sum"),
            fake=("fake", "sum"),
            hours=("hours", "sum"),
        )
        .reindex(SPLITS)
    )
    split_table["hours"] = split_table["hours"] / 3600
    split_table["fake_share"] = split_table["fake"] / split_table["clips"]
    split_table["clip_share"] = split_table["clips"] / n_clips
    split_table["fake_alloc_share"] = split_table["fake"] / n_fake_total
    split_table["clip_dev_pp"] = 100 * (split_table["clip_share"] - SPLIT_TARGET)
    split_table["fake_alloc_dev_pp"] = 100 * (split_table["fake_alloc_share"] - SPLIT_TARGET)
    return assignment, split_table, proposed, current_cost


def propose_label_stratified_split(
    frame: pd.DataFrame, seed: int = SEED
) -> tuple[pd.DataFrame, dict]:
    """Fallback: seeded label-stratified 80/10/10 clip split (no speaker grouping)."""
    rng = np.random.default_rng(seed)
    indices = {split: [] for split in SPLITS}
    for label in LABEL_ORDER:
        label_indices = frame.index[frame["label"] == label].to_numpy()
        rng.shuffle(label_indices)
        n_label = len(label_indices)
        n_train = int(round(SPLIT_TARGET[0] * n_label))
        n_val = int(round(SPLIT_TARGET[1] * n_label))
        indices["train"].extend(label_indices[:n_train].tolist())
        indices["val"].extend(label_indices[n_train : n_train + n_val].tolist())
        indices["test"].extend(label_indices[n_train + n_val :].tolist())

    rows = []
    for split in SPLITS:
        subset = frame.loc[indices[split]]
        rows.append(
            {
                "split": split,
                "speakers": subset["speaker"].nunique() if speaker_column else pd.NA,
                "clips": len(subset),
                "real": int((subset["label"] == "real").sum()),
                "fake": int((subset["label"] == "fake").sum()),
                "hours": float(subset["duration_proxy_s"].sum()),
            }
        )
    split_table = pd.DataFrame(rows).set_index("split").reindex(SPLITS)
    split_table["hours"] = split_table["hours"] / 3600
    split_table["fake_share"] = split_table["fake"] / split_table["clips"]
    split_table["clip_share"] = split_table["clips"] / len(frame)
    split_table["fake_alloc_share"] = split_table["fake"] / max(int((frame["label"] == "fake").sum()), 1)
    split_table["clip_dev_pp"] = 100 * (split_table["clip_share"] - SPLIT_TARGET)
    split_table["fake_alloc_dev_pp"] = 100 * (split_table["fake_alloc_share"] - SPLIT_TARGET)
    return split_table, indices


if speaker_column:
    split_assignment, split_table, proposed_speakers, split_objective = propose_speaker_disjoint_split(
        manifest, SEED
    )
    split_mode = "speaker-disjoint"
    speaker_lists = {
        split: sorted(proposed_speakers.index[proposed_speakers["split"] == split])
        for split in SPLITS
    }
    print("Speaker-disjoint proposal (seed 42):")
    for split in SPLITS:
        names = speaker_lists[split]
        print(f"  {split:5s} ({len(names):2d} speakers): {', '.join(names)}")
    assert set(split_assignment) == set(proposed_speakers.index)
    assert int(split_table["clips"].sum()) == n_total
    assert all(int(split_table.loc[split, "speakers"]) > 0 for split in SPLITS)
    split_speaker_overlap_check = sum(
        len(set(speaker_lists[first]) & set(speaker_lists[second]))
        for index, first in enumerate(SPLITS)
        for second in SPLITS[index + 1 :]
    )
else:
    split_table, fallback_indices = propose_label_stratified_split(manifest, SEED)
    split_mode = "label-stratified fallback"
    split_speaker_overlap_check = None

split_table = split_table[
    [
        "speakers",
        "clips",
        "real",
        "fake",
        "hours",
        "fake_share",
        "clip_share",
        "fake_alloc_share",
        "clip_dev_pp",
        "fake_alloc_dev_pp",
    ]
]
split_speaker_overlap = split_speaker_overlap_check if speaker_column else None
leakage_table = pd.DataFrame(
    [
        {
            "check": "Exact duplicate fingerprints (CRC32 + uncompressed size), full corpus",
            "observed": f"{len(duplicate_fingerprints):,} groups / {n_duplicate_files:,} files",
            "status": "PASS" if len(duplicate_fingerprints) == 0 else "REVIEW",
        },
        {
            "check": "Cross-label duplicate fingerprints",
            "observed": f"{len(cross_label_fingerprints):,} groups / {n_cross_label_duplicate_files:,} files",
            "status": "PASS" if len(cross_label_fingerprints) == 0 else "FAIL",
        },
        {
            "check": "MD5 duplicates within decoded sample",
            "observed": f"{md5_duplicate_groups:,} groups / {md5_duplicate_files:,} files",
            "status": "PASS" if md5_duplicate_groups == 0 else "REVIEW",
        },
        {
            "check": "Duplicate basenames",
            "observed": f"{basename_duplicates:,}",
            "status": "PASS" if basename_duplicates == 0 else "FAIL",
        },
        {
            "check": "Same-size cross-label collisions (heuristic only)",
            "observed": f"{size_collision_groups:,} sizes / {size_collision_files:,} files",
            "status": "INFO",
        },
        {
            "check": "Filename-pattern overlap across labels",
            "observed": f"{pattern_overlap:,} / {len(pattern_label_counts):,} patterns ({pattern_overlap_pct:.0f}%)",
            "status": "INFO",
        },
        {
            "check": "Speaker overlap across labels",
            "observed": f"{n_speakers_both:,} / {n_speakers:,} speakers",
            "status": "REVIEW" if n_speakers_both else "INFO",
        },
        {
            "check": "Speaker overlap across proposed splits",
            "observed": "0" if split_speaker_overlap == 0 else str(split_speaker_overlap),
            "status": "PASS" if split_speaker_overlap == 0 else ("INFO" if split_speaker_overlap is None else "FAIL"),
        },
        {
            "check": "Source / video ID available for source-level overlap audit",
            "observed": "no",
            "status": "REVIEW",
        },
        {
            "check": "Official train/dev/test protocol available",
            "observed": "no",
            "status": "INFO",
        },
    ]
)
show_table(leakage_table, "Table 6.2 — Leakage and duplicate checks", precision=0)

In [ ]:
show_table(
    split_table,
    "Table 6.3 — 80/10/10 split proposal (speaker-disjoint when metadata is available)",
    precision=3,
)
max_clip_dev = float(split_table["clip_dev_pp"].abs().max())
max_fake_dev = float(split_table["fake_alloc_dev_pp"].abs().max())
print(
    f"Split mode: {split_mode} | max clip deviation: {max_clip_dev:.3f} pp "
    f"| max fake-allocation deviation: {max_fake_dev:.3f} pp"
)
print(
    "80/10/10 is a target, not an exact constraint: three speakers each exceed 10% of "
    "the corpus, so val/test necessarily contain smaller speakers."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
x_positions = np.arange(len(split_table))
axes[0].bar(
    x_positions,
    split_table["real"],
    color=LABEL_COLORS["real"],
    label="real",
)
axes[0].bar(
    x_positions,
    split_table["fake"],
    bottom=split_table["real"],
    color=LABEL_COLORS["fake"],
    label="fake",
)
for position, (split, row) in enumerate(split_table.iterrows()):
    axes[0].text(
        position,
        row["clips"] + n_total * 0.01,
        f"{int(row['clips']):,}\n({row['clip_share'] * 100:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=8,
    )
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels([f"{split}\n(target {SPLIT_TARGET[index] * 100:.0f}%)" for index, split in enumerate(split_table.index)])
axes[0].set_ylabel("Clips")
axes[0].set_ylim(0, split_table["clips"].max() * 1.28)
axes[0].set_title("(a) Clip counts by proposed split")
axes[0].legend(title="Label")

bars = axes[1].bar(
    x_positions,
    split_table["fake_share"] * 100,
    color=LABEL_COLORS["fake"],
    width=0.55,
)
axes[1].axhline(global_fake_share * 100, color="black", linestyle="--", linewidth=1, label="global fake share")
for bar, value in zip(bars, split_table["fake_share"] * 100):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.8,
        f"{value:.1f}%",
        ha="center",
        fontsize=9,
    )
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(split_table.index)
axes[1].set_ylim(0, 100)
axes[1].set_ylabel("Fake share (%)")
axes[1].set_title("(b) Label balance within each split")
axes[1].legend(loc="upper right")

fig.suptitle("Fig 6.2 — Proposed speaker-disjoint split balance")
plt.tight_layout()
plt.show()

**Takeaway.** Speaker metadata is present and complete: **54 names, all 54 contributing
both real and fake clips**, with a median of ~213 clips per speaker. Exact-duplicate
checks over the full corpus (CRC32 + uncompressed size) and MD5 checks over the decoded
sample find no duplicate files; basenames are unique. Filename stems are purely numeric,
so they carry no source information, and `meta.csv` has no video/source ID: source-level
overlap **cannot be audited** and remains a documented limitation. Because every speaker
appears in both labels, a random clip split would leak speaker identity; the proposed
**speaker-disjoint 80/10/10 split** assigns each speaker to exactly one split, achieves
the target closely at the clip level, and preserves the global fake share. The three
largest speakers each exceed 10% of the corpus, so they must go to train; val/test
contain smaller speakers. This is a proposal only — no split column is exported.

## 7. Conclusions and requirement checklist

In [ ]:
runtime_min = (time.perf_counter() - START_TIME) / 60
codec_summary = ", ".join(sorted(audio_sample["codec"].dropna().unique()))
display(
    Markdown(
        f"""### Executive summary

- **Data source:** `{HF_REPO_ID}` at pinned revision `{HF_REVISION[:8]}` — one
  {human_bytes(archive_bytes)} ZIP archive, **{n_total:,} clips** ({n_real:,} real,
  {n_fake:,} fake), no extraction.
- **Label mix:** {real_pct:.1f}% real vs. {fake_pct:.1f}% fake. A majority-class baseline
  already reaches {majority_acc:.1f}% accuracy, so report balanced accuracy, per-class
  recall, AUC and EER instead.
- **Speakers:** {n_speakers} names in the released `meta.csv` (paper: {PAPER_SPEAKERS});
  all {n_speakers_both} observed speakers contribute both labels. Clips per speaker range
  from {speaker_clip_counts.min():,} to {speaker_clip_counts.max():,} (median
  {speaker_clip_counts.median():,.1f}); the top 3 speakers account for {top3_share * 100:.1f}%
  of the corpus.
- **Audio:** sampled files are {codec_summary}, 16 kHz mono; no decode errors. The
  full-manifest size proxy gives a mean duration of {proxy_mean_duration:.2f} s, median
  {proxy_median_duration:.2f} s, p95 {proxy_p95_duration:.2f} s, max
  {manifest['duration_proxy_s'].max():.2f} s, and **{proxy_total_hours:.2f} h total**
  ({proxy_real_hours:.2f} h real / {proxy_fake_hours:.2f} h fake), matching the published
  {PUBLISHED_TOTAL_HOURS:.1f} h. Proxy fit R² = {fit_r2:.6f}, sample MAE = {proxy_mae:.3f} s.
- **Quality and leakage:** labels and speaker names are complete; filenames are unique and
  dense; no exact duplicate fingerprints were found in the full corpus
  ({n_duplicate_files:,} files in duplicate groups; MD5 duplicate groups in sample:
  {md5_duplicate_groups}). Source/video IDs are absent, so source-level overlap cannot be
  audited. Speaker overlap across labels is {n_speakers_both}/{n_speakers}, which makes a
  random split leaky.
- **Split:** `{split_mode}` 80/10/10 proposal; max clip deviation
  {max_clip_dev:.3f} pp and max fake-allocation deviation {max_fake_dev:.3f} pp. This is a
  proposal only, not an executed split.
- **Scope:** eval-only benchmark, one undisclosed commercial TTS, no attack-ID axis, no
  official protocol, no model training or EER experiments (later PyTorch milestones).
"""
    )
)

In [ ]:
checklist = pd.DataFrame(
    [
        {
            "req": 1,
            "requirement": "Detailed dataset description",
            "section": "1–2",
            "evidence": "Tables 1.1–1.3, 2.1–2.3; Figs 2.1–2.2",
            "verdict": "PASS",
        },
        {
            "req": 2,
            "requirement": "Dataset quality assessment",
            "section": "1, 3, 6",
            "evidence": "Metadata completeness (1.2), integrity (1.3), format/quality (3.2), duplicates (6.2); Figs 3.2–3.3",
            "verdict": "PASS (sampled audio)",
        },
        {
            "req": 3,
            "requirement": "Adequacy for the chosen problem",
            "section": "2, 6",
            "evidence": "Corpus at a glance (2.1), speaker concentration (2.2, 6.1), split proposal (6.3); Figs 2.2, 6.1",
            "verdict": "PASS with caveats",
        },
        {
            "req": 4,
            "requirement": "Data augmentation options",
            "section": "5",
            "evidence": "Prioritised decision table (5.1); priority/risk figure (5.1)",
            "verdict": "PASS",
        },
        {
            "req": 5,
            "requirement": "Training / validation / test strategy",
            "section": "6",
            "evidence": "Speaker-disjoint 80/10/10 proposal (6.3); balance figure (6.2); no execution",
            "verdict": "PASS (proposal)",
        },
        {
            "req": 6,
            "requirement": "No data leakage across splits",
            "section": "6",
            "evidence": "Duplicate/leakage checks (6.2); speaker overlap figure (6.1); zero speaker overlap in proposal",
            "verdict": "PASS with source-provenance caveat",
        },
        {
            "req": 7,
            "requirement": "Audio quality and sampling consistency",
            "section": "3–4",
            "evidence": "Duration/signal stats (3.1), quality checks (3.2), representativeness (3.4); Figs 3.1–3.4, 4.1",
            "verdict": "PASS",
        },
    ]
)
show_table(checklist, "Table 7.1 — Requirement coverage checklist", precision=0)

In [ ]:
export_columns = [
    "file_name",
    "file_index",
    "member_path",
    "speaker",
    "label",
    "target",
    "file_size",
    "compressed_size",
    "crc32",
    "duration_proxy_s",
    "extension",
]
manifest_path = WORK_DIR / "inthewild_manifest.csv"
manifest[export_columns].to_csv(manifest_path, index=False)
print(f"Manifest written to: {manifest_path} ({manifest_path.stat().st_size / 1e6:.2f} MB)")
print(f"Rows exported: {len(manifest):,}")
print(f"Notebook runtime: {runtime_min:.1f} min")
display(manifest[export_columns].head(6))

**Final conclusion.** In-The-Wild is a **wild but metadata-rich eval-only** corpus:
31,779 clips (62.8% real / 37.2% fake), 37.9 h, 54 released speaker names, and uniform
16 kHz mono WAV / float32 audio. Its difficulty is not container inconsistency or
missing labels — both are clean — but **content diversity, speaker concentration, a
single undisclosed TTS, source provenance that cannot be audited, and the absence of an
official split**. All 54 speakers appear in both labels, so a random clip split would
leak speaker identity; the proposed speaker-disjoint 80/10/10 split is the minimum
defensible protocol for the next milestone, with source-level overlap documented as a
residual risk. Compared with ASVspoof 2019 LA eval (89.7% spoof, 13 balanced attacks,
official protocol, lab-clean audio), In-The-Wild flips the imbalance direction, removes
the attack-ID axis, adds wild channels and speaker skew, and requires us to define the
split ourselves. All outputs are seeded with 42 and the exported manifest is the bridge
to the PyTorch training milestone.